# Linear Regression Example (Wine Quality Dataset)

Here it is demonstrated how to use the `LinearRegression` module from the CMOR-438 library to predict wine quality scores.
In this example, the Wine Quality dataset is used to train, test, and evaluate three fitting methods.

**Goal: Predict the quality score of red wine based on its physicochemical properties.**

The Wine Quality dataset has:
- **Samples:** 1,143 red wine observations
- **Features:** 11 physicochemical measurements (alcohol, acidity, sulphates, etc.)
- **Target:** Quality score from 3 to 8 (continuous — treated as regression)

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# NOTEBOOK_DIR resolves to the folder this notebook lives in.
# Jupyter sets the working directory to wherever it was launched from,
# so we use __file__ would not work — os.path.abspath('') is the correct
# approach for Jupyter notebooks.
# Find the repo root reliably on any machine.
# We search upward from the current working directory until we find
# the 'data' folder, which only exists at the repo root.
def _find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(10):  # search up to 10 levels up
        if os.path.isdir(os.path.join(path, 'data')) and os.path.isdir(os.path.join(path, 'src')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not find repo root. Make sure you launched Jupyter from inside the CMOR-438 folder."
    )

REPO_ROOT = _find_repo_root()

# Algorithm source files live in src/supervised/ at the repo root,
# which is four levels up from examples/supervised/<algo>/
SRC_SUP  = os.path.join(REPO_ROOT, 'src', 'supervised')
sys.path.insert(0, SRC_SUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(REPO_ROOT, 'data')
from linear_regression import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv(os.path.join(DATA_DIR, 'WineQT.csv')).drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']

print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")
print(f"Features: {FEATURE_COLS}")
print(f"Target range: {wine['quality'].min()} – {wine['quality'].max()}")

## 2. Preprocessing

Standardise features to zero mean and unit variance, then split into 80% train / 20% test.

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y = wine['quality'].values.astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_tr.shape[0]}")
print(f"Test samples:     {X_te.shape[0]}")

## 3. Train — Three Fitting Methods

Three variants of LinearRegression are trained and compared:
- **OLS** — exact closed-form solution via the normal equation
- **Ridge** — OLS with L2 regularisation (alpha=1.0) to prevent overfitting
- **GD** — iterative Gradient Descent over 2,000 steps

In [ ]:
results = {}
for method in ('ols', 'ridge', 'gd'):
    m = LinearRegression(method=method, learning_rate=0.01, n_iterations=2000)
    m.fit(X_tr, y_tr)
    results[method] = {'R2': m.score(X_te, y_te), 'MSE': m.mse(X_te, y_te), 'model': m}
    print(f'{method.upper():>5s} | R²={results[method]["R2"]:.4f}  MSE={results[method]["MSE"]:.4f}')

## 4. Results and Visualisation

Three plots are produced:
- **OLS predicted vs actual** — shows where the model is accurate and where it struggles
- **Ridge predicted vs actual** — comparison to OLS
- **GD loss curve** — shows the gradient descent solver converging

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, method in zip(axes[:2], ('ols', 'ridge')):
    preds = results[method]['model'].predict(X_te)
    ax.scatter(y_te, preds, alpha=0.45, s=18, color='steelblue')
    lo, hi = y_te.min()-0.2, y_te.max()+0.2
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect fit')
    ax.set_xlabel('Actual Quality'); ax.set_ylabel('Predicted Quality')
    ax.set_title(f'Linear Reg ({method.upper()})  R²={results[method]["R2"]:.3f}', fontweight='bold')
    ax.legend(fontsize=8)

axes[2].plot(results['gd']['model'].loss_history_, color='darkorange', lw=1.5)
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('MSE Loss')
axes[2].set_title('Gradient Descent Loss Curve', fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Analysis

**Results summary:**

| Method | R² | MSE |
|---|---|---|
| OLS | 0.3171 | 0.3800 |
| Ridge | 0.3173 | 0.3799 |
| GD | 0.3170 | 0.3801 |

All three methods produce nearly identical results, which is expected — on this dataset size, all three converge to essentially the same solution. The near-zero difference between OLS and Ridge confirms there is no meaningful multicollinearity problem in the features.

**R² of ~0.32** means the model explains about 32% of the variance in wine quality. This is modest but not surprising — quality scores are highly subjective and influenced by factors not captured in physicochemical measurements alone. The model does better than random but leaves substantial unexplained variance.

**The GD loss curve** should show a smooth downward trajectory, confirming the learning rate of 0.01 is appropriate. If it were jagged or flat the learning rate would need adjustment.

**The predicted vs actual scatter plots** will show a diagonal cloud centred on the perfect-fit line, but with considerable vertical spread — predictions cluster around scores 5 and 6 (the most common quality values) and struggle at the extremes (3–4 and 8), which are underrepresented in the training data.

**Key takeaway:** Linear regression is a reasonable baseline for this task but the relationship between physicochemical features and perceived quality is genuinely non-linear, which explains the modest R². Tree-based models handle this better, as seen in the Decision Tree and Ensemble examples.